# Threads, Processes and the GIL

Companion notebook for the [Why Concurrency Is Hard](https://codemia.io/courses/concurrency_and_async_programming/cc_why_concurrency_is_hard) and [Threads and the Scheduler](https://codemia.io/courses/concurrency_and_async_programming/cc_threads_and_the_scheduler) lessons on Codemia.

**What you will measure**: the same CPU bound job run sequentially, on a thread pool and on a process pool; an I/O bound job run the same three ways; a lost update race in pure Python, and the lock that fixes it; and a CPU heavy library call that releases the GIL, so threads scale after all. Every number comes from your own machine, so read the printed output rather than trusting the prose.

## Setup

In [ ]:
%pip install -q matplotlib

In [ ]:
import os
import sys
import threading
import time
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor

import matplotlib.pyplot as plt

cores = os.cpu_count()
gil = getattr(sys, "_is_gil_enabled", lambda: True)()
print(f"Python {sys.version.split()[0]}, {cores} cores, GIL enabled: {gil}")

`sys._is_gil_enabled()` exists from Python 3.13. On a normal build it returns `True`. On the free threaded build (`python3.13t`, officially supported from 3.14) it returns `False`, and the CPU bound thread results below change completely. Try it there once you have run this notebook on a normal build.

## 1. Two workloads

Process pools pickle the function they run and look it up by module name in the worker, so the workloads live in a small module file instead of a notebook cell. `cpu_work` is a pure Python loop that never releases the GIL. `io_work` sleeps, and `time.sleep` releases the GIL while it waits, exactly like a socket read would.

In [ ]:
with open("workloads.py", "w") as f:
    f.write('''import time

def cpu_work(n):
    total = 0
    for i in range(n):
        total += i * i % 7
    return total

def io_work(seconds):
    time.sleep(seconds)
    return seconds
''')

import importlib
import workloads
importlib.reload(workloads)
from workloads import cpu_work, io_work

t0 = time.perf_counter()
cpu_work(3_000_000)
one = time.perf_counter() - t0
print(f"one cpu_work call: {one:.2f}s")

## 2. CPU bound: threads do not help

Run four equal chunks of CPU work three ways. With the GIL, only one thread executes Python bytecode at a time, so the thread pool takes about as long as running the chunks one after another. Expect a speedup somewhere around 1x to 1.5x, depending on how the threads hand the GIL back and forth. Processes each have their own interpreter and their own GIL, so they run in parallel, minus the cost of starting workers and pickling arguments.

In [ ]:
def timed(fn):
    t0 = time.perf_counter()
    fn()
    return time.perf_counter() - t0

N, TASKS = 3_000_000, 4
workers = max(1, min(TASKS, cores))

cpu = {
    "sequential": timed(lambda: [cpu_work(N) for _ in range(TASKS)]),
    "threads": timed(lambda: list(ThreadPoolExecutor(workers).map(cpu_work, [N] * TASKS))),
    "processes": timed(lambda: list(ProcessPoolExecutor(workers).map(cpu_work, [N] * TASKS))),
}
for k, v in cpu.items():
    print(f"{k:11s} {v:5.2f}s   speedup {cpu['sequential'] / v:4.2f}x")

On a machine with at least four cores, expect a thread speedup that stays well below 2x and a process speedup well above 2x. Colab usually gives you two cores, so processes top out near 2x there.

## 3. I/O bound: threads are fine

Now twenty tasks that each wait 0.2 seconds. Waiting does not need the GIL, so twenty threads overlap all their waits and finish in roughly one wait. A process pool also overlaps them, but only as many at a time as it has workers, and each worker costs a whole interpreter.

In [ ]:
WAITS = [0.2] * 20

io = {
    "sequential": timed(lambda: [io_work(s) for s in WAITS]),
    "threads": timed(lambda: list(ThreadPoolExecutor(20).map(io_work, WAITS))),
    "processes": timed(lambda: list(ProcessPoolExecutor(workers).map(io_work, WAITS))),
}
for k, v in io.items():
    print(f"{k:11s} {v:5.2f}s   speedup {io['sequential'] / v:4.2f}x")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
for ax, (title, res) in zip(axes, [("CPU bound (4 chunks)", cpu), ("I/O bound (20 waits of 0.2s)", io)]):
    ax.bar(list(res), list(res.values()), color=["#94a3b8", "#3b82f6", "#10b981"])
    ax.set_title(title)
    ax.set_ylabel("seconds")
plt.tight_layout()
plt.show()

## 4. The GIL does not make your code thread safe

The GIL protects the interpreter's own data structures. It says nothing about your invariants. A read, a bit of work, then a write is three separate steps, and the interpreter may switch threads between any of them. Lowering the switch interval makes those switches frequent enough to see within a second.

In [ ]:
class Account:
    def __init__(self):
        self.balance = 0

def add_fee(v):
    return v + 1

def unsafe_deposit(acct, times):
    for _ in range(times):
        v = acct.balance       # read
        v = add_fee(v)         # work
        acct.balance = v       # write

def run(target, threads=4, times=100_000):
    acct = Account()
    ts = [threading.Thread(target=target, args=(acct, times)) for _ in range(threads)]
    for t in ts:
        t.start()
    for t in ts:
        t.join()
    return acct.balance, threads * times

old = sys.getswitchinterval()
sys.setswitchinterval(1e-6)
got, want = run(unsafe_deposit)
print(f"unsafe: balance {got:,} of {want:,}, lost {want - got:,} updates")

Every lost update is one thread reading the balance, being paused, and then writing back a value computed from that stale read, overwriting the deposits another thread made in between. A lock turns the three steps into one critical section.

In [ ]:
lock = threading.Lock()

def safe_deposit(acct, times):
    for _ in range(times):
        with lock:
            v = acct.balance
            v = add_fee(v)
            acct.balance = v

got, want = run(safe_deposit)
print(f"locked: balance {got:,} of {want:,}, lost {want - got:,} updates")
sys.setswitchinterval(old)

## 5. Code that releases the GIL scales on threads

C extensions can release the GIL around long computations that do not touch Python objects. `hashlib` does so for inputs larger than about 2 KB, as do zlib, most of NumPy and many I/O libraries. Hashing large buffers on a thread pool therefore runs in parallel even on a normal build.

In [ ]:
import hashlib

blob = os.urandom(128 * 1024 * 1024)

def digest(_):
    return hashlib.sha256(blob).hexdigest()

seq = timed(lambda: [digest(i) for i in range(TASKS)])
thr = timed(lambda: list(ThreadPoolExecutor(workers).map(digest, range(TASKS))))
print(f"sha256 sequential {seq:.2f}s, threads {thr:.2f}s, speedup {seq / thr:.2f}x")

With four or more cores the thread speedup here is well above 1x, unlike the pure Python loop in section 2. The rule of thumb that follows: threads in Python help for I/O and for library calls that release the GIL; for pure Python CPU work, use processes, or a free threaded build.

## 6. Exercises

1. Change `TASKS` to `2 * cores` in section 2. Does the process speedup keep growing? Why not?
2. Remove the `add_fee` call from `unsafe_deposit` and use `acct.balance += 1`. On Python 3.10 and later the race becomes far rarer. Find out why by reading about when CPython checks for a pending thread switch.
3. Time `ProcessPoolExecutor` on a single tiny task to measure the cost of starting a worker, and work out how much CPU a task needs before a process pool pays off.
4. If you have Python 3.13t or 3.14t, rerun the notebook and compare the thread speedup in section 2. Then check whether section 4 still loses updates without the lock.